# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.crm_prd_info')

In [0]:
df.display()

# Silver Transformations

- Trimming any space before or after 

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Product Key Parsing
- 1st code -> Takes the first 5 letter and replaces the '-' to '_' and stores into a new column called 'cat_id'

- 2nd code -> Chops off the part and works on the remainder of the part. Starts extracting from the 7th character. Then removes the first 6 charactes (5 characters plus the extra separator character)

In [0]:
# 1st 
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
# 2nd
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

# Cost Clean up -> Null Safe.
### What the below code is doing.
- The coalesce function works like a priority list. It looks at the arguments you give it from left to right and returns the first value that isn't null.
1. Looks at the prd_cost column first. 
2. If it has a value, it keeps it.
3. If its null, it moves to next item and fits 0.

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

In [0]:
df.display()

# Product Line Normalisation
- Converting M,R,S and T to its original naming convention

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

# Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

# Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Writing into Silver Table

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.crm_products')

In [0]:
%sql
select * from workspace.silver.crm_products limit 10